In [ ]:
# === Colab bootstrap (no-op outside Colab) ===========================
# Clones the repo, installs minimal deps, mounts Google Drive, and
# symlinks heavy assets from Drive into the paths the notebook uses.
# See COLAB_SETUP.md for details.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import os, subprocess, sys
    REPO = "/content/INF8225_Projet"
    if not os.path.isdir(REPO):
        subprocess.check_call([
            "git", "clone", "--depth", "1", "--branch", "temp",
            "https://github.com/moradBMH/INF8225_Projet.git", REPO,
        ])
    if os.getcwd() != REPO:
        os.chdir(REPO)
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    from colab.setup import setup
    setup()
# ======================================================================

In [ ]:
import os

# Remonte l'arborescence jusqu'à trouver le dossier racine (qui contient 'data' ou 'MedSAM')
while not os.path.isdir('data') and os.getcwd() != os.path.abspath(os.sep):
    os.chdir('..')
print("Dossier de travail actuel :", os.getcwd())

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
import torchvision
from skimage import io, transform
from tqdm import tqdm

# Imports Grounding DINO
from mmdet.apis import init_detector, inference_detector
from mmdet.utils import register_all_modules

# Imports MedSAM
from MedSAM.segment_anything import sam_model_registry
from MedSAM.MedSAM_Inference import medsam_inference

from utils import calculate_dice, load_rgb_image, show_segmentation
from colab.drive_paths import output_dir

In [ ]:
# ==========================================
# 1. CONFIGURATION DES CHEMINS
# ==========================================
base_img_folder = "data/Kvasir-SEG/images"
base_mask_folder = "data/Kvasir-SEG/masks"
test_json_path = "data/Kvasir-SEG/test.json"

# Hiérarchie des dossiers de résultats
results_folder = output_dir("kvasir_implementation", "test_kvasir")
metrics_folder = output_dir("kvasir_implementation", "test_kvasir", "metrics")
output_masks_folder = output_dir(
    "kvasir_implementation", "test_kvasir", "masks", "zero_shot_segmentation_masks_kvasir"
)
csv_output_path = metrics_folder / "dice_zero_shot_kvasir.csv"

# ==========================================
# 2. VÉRIFICATION DE L'EXISTENCE DES RÉSULTATS
# ==========================================
if os.path.exists(csv_output_path):
    print(f"Les résultats existent déjà dans {csv_output_path}.")
    print("Inférence ignorée. Chargement des données existantes...")
    df = pd.read_csv(csv_output_path)
    
else:
    # ==========================================
    # 3. INITIALISATION DES MODÈLES (Si inférence requise)
    # ==========================================
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Appareil détecté : {device}")

    register_all_modules()
    dino_config = 'models_weights/grounding_dino_swin-t_pretrain_obj365_goldg.py'
    dino_checkpoint = 'models_weights/grounding_dino_swin-t_pretrain_obj365_goldg_20231122_132602-4ea751ce.pth'
    dino_model = init_detector(dino_config, dino_checkpoint, device=device)

    medsam_checkpoint = "MedSAM/work_dir/MedSAM/medsam_vit_b.pth"
    medsam_model = sam_model_registry["vit_b"](checkpoint=medsam_checkpoint).to(device)
    medsam_model.eval()

    # ==========================================
    # 4. BOUCLE D'INFÉRENCE
    # ==========================================
    with open(test_json_path, 'r') as f:
        test_data = json.load(f)

    test_images_list = test_data['images'] 
    dice_list = []

    print(f"\nDébut de l'évaluation sur les {len(test_images_list)} images du fichier test.json...")

    for img_info in tqdm(test_images_list, desc="Inférence Test Set"):
        file_name = img_info['file_name']
        img_id = img_info['id']
        
        img_path = os.path.join(base_img_folder, file_name)
        mask_path = os.path.join(base_mask_folder, file_name)

        if not os.path.exists(img_path):
            continue
            
        img_3c = load_rgb_image(img_path)
        H, W, _ = img_3c.shape

        true_seg = io.imread(mask_path)
        if len(true_seg.shape) == 3:
            true_seg = true_seg[:,:,0]
        true_seg = (true_seg > 0).astype(np.uint8)

        full_medsam_seg = np.zeros((H, W), dtype=np.uint8)

        with torch.no_grad():
            result = inference_detector(dino_model, img_path, text_prompt="polyp.")
            pred_instances = result.pred_instances
            
            scores = pred_instances.scores
            bboxes = pred_instances.bboxes
            
            mask_conf = scores > 0.05 
            valid_boxes = bboxes[mask_conf]
            valid_scores = scores[mask_conf]

            if len(valid_boxes) > 0:
                keep_indices = torchvision.ops.nms(valid_boxes, valid_scores, iou_threshold=0.5)
                final_boxes = valid_boxes[keep_indices].cpu().numpy()
                
                img_1024 = transform.resize(img_3c, (1024, 1024), order=3, preserve_range=True, anti_aliasing=True).astype(np.uint8)
                img_1024 = (img_1024 - img_1024.min()) / np.clip(img_1024.max() - img_1024.min(), a_min=1e-8, a_max=None)
                img_1024_tensor = torch.tensor(img_1024).float().permute(2, 0, 1).unsqueeze(0).to(device)
                
                image_embedding = medsam_model.image_encoder(img_1024_tensor)
                
                for box in final_boxes:
                    box_np = np.array([box]) 
                    box_1024 = box_np / np.array([W, H, W, H]) * 1024
                    medsam_seg = medsam_inference(medsam_model, image_embedding, box_1024, H, W)
                    full_medsam_seg[medsam_seg > 0] = 1

        # Sauvegarde du masque prédit
        io.imsave(
            os.path.join(output_masks_folder, "seg_" + os.path.basename(img_path)),
            (full_medsam_seg*255).astype(np.uint8),
            check_contrast=False,
        )

        # Calcul et stockage du DICE
        dice_score = calculate_dice(true_seg, full_medsam_seg)
        dice_list.append({
            "image_id": img_id,
            "file_name": file_name,
            "dice": dice_score
        })

    # Sauvegarde finale
    df = pd.DataFrame(dice_list)
    df.to_csv(csv_output_path, index=False)
    print(f"\nRésultats sauvegardés dans : {csv_output_path}")

# ==========================================
# 5. ANALYSE DES RÉSULTATS
# ==========================================
print("\n" + "="*40)
print(f"DICE MOYEN : {df['dice'].mean():.4f}")
print(f"DICE MEDIAN : {df['dice'].median():.4f}")
print(f"MEILLEUR DICE : {df['dice'].max():.4f}")
print(f"PIRE DICE : {df['dice'].min():.4f}")
print("="*40)

In [ ]:
# 1. Chargement des bboxes (Nécessaire pour la fonction show_segmentation)
bboxes_path = "data/Kvasir-SEG/kavsir_bboxes.json"
with open(bboxes_path, 'r') as f:
    bboxes = json.load(f)

# 2. Extraction des extrêmes depuis le DataFrame
# nlargest et nsmallest sont des fonctions Pandas très optimisées pour ça
top_3_best = df.nlargest(3, 'dice')
top_5_worst = df.nsmallest(5, 'dice')

# Assurez-vous que les fonctions show_box, show_mask et show_segmentation 
# sont bien définies ou importées avant d'exécuter cette partie.

print("\n" + "="*40)
print("=== AFFICHAGE DES 3 MEILLEURES PRÉDICTIONS ===")
for index, row in top_3_best.iterrows():
    print(f"Image: {row['file_name']} - DICE: {row['dice']:.4f}")
    show_segmentation(
        img_name=row['file_name'],
        img_folder=base_img_folder,
        masks_folder=base_mask_folder,
        output_masks_folder=output_masks_folder,
        bboxes=bboxes
    )

print("\n" + "="*40)
print("=== AFFICHAGE DES 5 PIRES PRÉDICTIONS ===")
for index, row in top_5_worst.iterrows():
    print(f"Image: {row['file_name']} - DICE: {row['dice']:.4f}")
    show_segmentation(
        img_name=row['file_name'],
        img_folder=base_img_folder,
        masks_folder=base_mask_folder,
        output_masks_folder=output_masks_folder,
        bboxes=bboxes
    )

In [ ]:
# ==========================================
# 1. CONFIGURATION DES CHEMINS
# ==========================================
base_img_folder = "data/Kvasir-SEG/images"
base_mask_folder = "data/Kvasir-SEG/masks"
test_json_path = "data/Kvasir-SEG/test.json"

# Hiérarchie des dossiers de résultats
results_folder = output_dir("kvasir_implementation", "test_kvasir")
metrics_folder = output_dir("kvasir_implementation", "test_kvasir", "metrics")
output_masks_folder = output_dir(
    "kvasir_implementation", "test_kvasir", "masks", "fine_tuned_segmentation_masks_kvasir"
)

# ATTENTION : J'ai corrigé le nom ici pour ne pas écraser le zero-shot
csv_output_path = metrics_folder / "dice_fine_tuned_kvasir.csv"

# ==========================================
# 2. VÉRIFICATION DE L'EXISTENCE DES RÉSULTATS
# ==========================================
if os.path.exists(csv_output_path):
    print(f"Les résultats existent déjà dans {csv_output_path}.")
    print("Inférence ignorée. Chargement des données existantes...")
    df = pd.read_csv(csv_output_path)
    
else:
    # ==========================================
    # 3. INITIALISATION DES MODÈLES (Si inférence requise)
    # ==========================================
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Appareil détecté : {device}")

    register_all_modules()
    dino_config = 'kvasir_implementation/polyp_config_v2.py'
    dino_checkpoint = 'work_dirs/polyp_config_v2/best_coco_bbox_mAP_epoch_5.pth'
    dino_model = init_detector(dino_config, dino_checkpoint, device=device)

    medsam_checkpoint = "MedSAM/work_dir/MedSAM/medsam_vit_b.pth"
    medsam_model = sam_model_registry["vit_b"](checkpoint=medsam_checkpoint).to(device)
    medsam_model.eval()

    # ==========================================
    # 4. BOUCLE D'INFÉRENCE
    # ==========================================
    with open(test_json_path, 'r') as f:
        test_data = json.load(f)

    test_images_list = test_data['images'] 
    dice_list = []

    print(f"\nDébut de l'évaluation sur les {len(test_images_list)} images du fichier test.json...")

    for img_info in tqdm(test_images_list, desc="Inférence Test Set"):
        file_name = img_info['file_name']
        img_id = img_info['id']
        
        img_path = os.path.join(base_img_folder, file_name)
        mask_path = os.path.join(base_mask_folder, file_name)

        if not os.path.exists(img_path):
            continue
            
        img_3c = load_rgb_image(img_path)
        H, W, _ = img_3c.shape

        true_seg = io.imread(mask_path)
        if len(true_seg.shape) == 3:
            true_seg = true_seg[:,:,0]
        true_seg = (true_seg > 0).astype(np.uint8)

        full_medsam_seg = np.zeros((H, W), dtype=np.uint8)

        with torch.no_grad():
            result = inference_detector(dino_model, img_path, text_prompt="polyp.")
            pred_instances = result.pred_instances
            
            scores = pred_instances.scores
            bboxes = pred_instances.bboxes
            
            mask_conf = scores > 0.30 # Seuil optimisé pour fine-tuned
            valid_boxes = bboxes[mask_conf]
            valid_scores = scores[mask_conf]

            if len(valid_boxes) > 0:
                keep_indices = torchvision.ops.nms(valid_boxes, valid_scores, iou_threshold=0.5)
                final_boxes = valid_boxes[keep_indices].cpu().numpy()
                
                img_1024 = transform.resize(img_3c, (1024, 1024), order=3, preserve_range=True, anti_aliasing=True).astype(np.uint8)
                img_1024 = (img_1024 - img_1024.min()) / np.clip(img_1024.max() - img_1024.min(), a_min=1e-8, a_max=None)
                img_1024_tensor = torch.tensor(img_1024).float().permute(2, 0, 1).unsqueeze(0).to(device)
                
                image_embedding = medsam_model.image_encoder(img_1024_tensor)
                
                for box in final_boxes:
                    box_np = np.array([box]) 
                    box_1024 = box_np / np.array([W, H, W, H]) * 1024
                    medsam_seg = medsam_inference(medsam_model, image_embedding, box_1024, H, W)
                    full_medsam_seg[medsam_seg > 0] = 1

        # Sauvegarde du masque prédit
        io.imsave(
            os.path.join(output_masks_folder, "seg_" + os.path.basename(img_path)),
            (full_medsam_seg*255).astype(np.uint8),
            check_contrast=False,
        )

        # Calcul et stockage du DICE
        dice_score = calculate_dice(true_seg, full_medsam_seg)
        dice_list.append({
            "image_id": img_id,
            "file_name": file_name,
            "dice": dice_score
        })

    # Sauvegarde finale
    df = pd.DataFrame(dice_list)
    df.to_csv(csv_output_path, index=False)
    print(f"\nRésultats sauvegardés dans : {csv_output_path}")

# ==========================================
# 5. ANALYSE DES RÉSULTATS
# ==========================================
print("\n" + "="*40)
print(f"DICE MOYEN : {df['dice'].mean():.4f}")
print(f"DICE MEDIAN : {df['dice'].median():.4f}")
print(f"MEILLEUR DICE : {df['dice'].max():.4f}")
print(f"PIRE DICE : {df['dice'].min():.4f}")
print("="*40)

In [ ]:
# 1. Chargement des bboxes (Nécessaire pour la fonction show_segmentation)
bboxes_path = "data/Kvasir-SEG/kavsir_bboxes.json"
with open(bboxes_path, 'r') as f:
    bboxes = json.load(f)

# 2. Extraction des extrêmes depuis le DataFrame
# nlargest et nsmallest sont des fonctions Pandas très optimisées pour ça
top_3_best = df.nlargest(3, 'dice')
top_5_worst = df.nsmallest(5, 'dice')

# Assurez-vous que les fonctions show_box, show_mask et show_segmentation 
# sont bien définies ou importées avant d'exécuter cette partie.

print("\n" + "="*40)
print("=== AFFICHAGE DES 3 MEILLEURES PRÉDICTIONS ===")
for index, row in top_3_best.iterrows():
    print(f"Image: {row['file_name']} - DICE: {row['dice']:.4f}")
    show_segmentation(
        img_name=row['file_name'],
        img_folder=base_img_folder,
        masks_folder=base_mask_folder,
        output_masks_folder=output_masks_folder,
        bboxes=bboxes
    )

print("\n" + "="*40)
print("=== AFFICHAGE DES 5 PIRES PRÉDICTIONS ===")
for index, row in top_5_worst.iterrows():
    print(f"Image: {row['file_name']} - DICE: {row['dice']:.4f}")
    show_segmentation(
        img_name=row['file_name'],
        img_folder=base_img_folder,
        masks_folder=base_mask_folder,
        output_masks_folder=output_masks_folder,
        bboxes=bboxes
    )